<a href="https://colab.research.google.com/github/epleywin/ECON3916-Statistical-Machine-Learning/blob/main/Project%201/%20Phase_3_Epley_Win.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: Econometric Model of Resume Callback Outcomes

This notebook estimates an econometric model to examine which resume
characteristics are associated with employer callback decisions. The
goal is to move beyond exploratory analysis and estimate a baseline
regression model that can help explain variation in callback outcomes.

The analysis uses a Linear Probability Model estimated with the
statsmodels package. Robust standard errors are applied to account for
heteroskedasticity, which is common when the dependent variable is
binary.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
df = pd.read_csv("resume.csv")

df.head()

,job_ad_id,job_city,job_industry,job_type,job_fed_contractor,job_equal_opp_employer,job_ownership,job_req_any,job_req_communication,job_req_education,...,honors,worked_during_school,years_experience,computer_skills,special_skills,volunteer,military,employment_holes,has_email_address,resume_quality
0,384,Chicago,manufacturing,supervisor,NaN,1,unknown,1,0,0,...,0,0,6,1,0,0,0,1,0,low
1,384,Chicago,manufacturing,supervisor,NaN,1,unknown,1,0,0,...,0,1,6,1,0,1,1,0,1,high
2,384,Chicago,manufacturing,supervisor,NaN,1,unknown,1,0,0,...,0,1,6,1,0,0,0,0,0,low
3,384,Chicago,manufacturing,supervisor,NaN,1,unknown,1,0,0,...,0,0,6,1,1,1,0,1,1,high
4,385,Chicago,other_service,secretary,0.0,1,nonprofit,1,0,0,...,0,1,22,1,0,0,0,0,1,high


Each row in the dataset represents a resume submitted to a job
advertisement as part of a field experiment. The dataset contains
information about applicant characteristics, resume features, and
whether the employer responded with a callback.

## Research Question

The goal of this analysis is to estimate how certain resume
characteristics influence the probability that an applicant receives a
callback from an employer.

The dependent variable in the model is received_callback, which equals
1 if the employer contacted the applicant and 0 otherwise.

Several explanatory variables are included in the model to represent
resume quality, applicant background, and work experience.

In [3]:
# Create race dummy variable
df["race_black"] = (df["race"] == "black").astype(int)

# Create resume quality dummy
df["high_quality"] = (df["resume_quality"] == "high").astype(int)

Some categorical variables must be converted into numeric indicator
variables before they can be used in a regression model. The race
variable is converted into a binary variable that equals 1 if the
applicant is classified as black and 0 otherwise. Resume quality is
also converted into a binary variable indicating whether the resume is
classified as high quality.

## Baseline Model

To estimate the relationship between resume characteristics and
callback outcomes, a Linear Probability Model is estimated using
ordinary least squares.

The dependent variable is received_callback. The explanatory variables
include applicant race, resume quality, years of work experience, and
whether the applicant has a college degree.

Because the dependent variable is binary, the error variance is likely
to be heteroskedastic. To address this issue, heteroskedasticity robust
standard errors are used when estimating the model.

In [4]:
model = smf.ols(
    formula="received_callback ~ race_black + high_quality + years_experience + college_degree",
    data=df
).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      received_callback   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     8.178
Date:                Mon, 16 Mar 2026   Prob (F-statistic):           1.44e-06
Time:                        13:51:12   Log-Likelihood:                -551.91
No. Observations:                4870   AIC:                             1114.
Df Residuals:                    4865   BIC:                             1146.
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.0687      0.011  

## Interpretation of Coefficients

The coefficients in the Linear Probability Model represent changes in
the probability of receiving a callback from an employer. Because the
dependent variable is binary, each coefficient can be interpreted as a
change in callback probability associated with a one unit change in the
explanatory variable, holding other variables constant.

The coefficient on race_black is -0.0319. This means that resumes
associated with black applicants receive callbacks about 3.2
percentage points less often than resumes associated with white
applicants, ceteris paribus.

The coefficient on high_quality is 0.0112. This suggests that resumes
classified as high quality are associated with a 1.1 percentage point
increase in the probability of receiving a callback, holding other
factors constant. However, this estimate is not statistically
significant at conventional levels.

The coefficient on years_experience is 0.0032. This means that each
additional year of work experience is associated with about a 0.32
percentage point increase in the probability of receiving a callback,
ceteris paribus.

Finally, the coefficient on college_degree is small and not
statistically significant, suggesting that having a college degree
does not appear to strongly affect callback probability in this
baseline specification.

## Interpretation of the Interaction Model

The interaction model allows the effect of resume quality to differ
across racial groups by including an interaction term between race and
resume quality.

The coefficient on race_black in this specification is -0.0230. This
suggests that resumes associated with black applicants receive
callbacks about 2.3 percentage points less often than resumes
associated with white applicants, holding other variables constant.

The coefficient on high_quality is 0.0201. This means that high quality
resumes are associated with roughly a 2 percentage point increase in
the probability of receiving a callback for the baseline group.

The interaction term race_quality_interaction has a coefficient of
-0.0177. This suggests that the effect of resume quality may be smaller
for black applicants relative to white applicants. However, the
interaction coefficient is not statistically significant, which means
the data does not provide strong evidence that the effect of resume
quality differs across groups.

Overall, the results suggest that work experience is consistently
associated with higher callback probabilities, while the estimated
effects of resume quality and college education are less precise in
this specification.

In [5]:
interaction_model = smf.ols(
    formula="received_callback ~ race_black * high_quality + years_experience + college_degree",
    data=df
).fit(cov_type="HC1")

print(interaction_model.summary())

                            OLS Regression Results                            
Dep. Variable:      received_callback   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     6.695
Date:                Mon, 16 Mar 2026   Prob (F-statistic):           3.18e-06
Time:                        13:51:12   Log-Likelihood:                -551.26
No. Observations:                4870   AIC:                             1115.
Df Residuals:                    4864   BIC:                             1153.
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Economic Mechanism

Employers typically receive many job applications for each open
position and must quickly decide which applicants to contact for
interviews. Because employers have limited information about each
candidate, they rely on signals contained in resumes when making
these decisions.

Characteristics such as work experience or resume quality may act as
signals of productivity or preparedness for the job. Applicants with
more experience may therefore be more likely to receive callbacks
because employers interpret experience as evidence of job readiness.

At the same time, employers may interpret certain applicant
characteristics differently when evaluating resumes. If employers rely
on assumptions or expectations about different groups of applicants,
these perceptions may influence callback decisions even when resumes
contain similar qualifications.

The regression results suggest that work experience is positively
associated with callback probability, while resumes associated with
black applicants receive callbacks at a lower rate on average,
holding other resume characteristics constant.

## Potential Omitted Variable Bias

One potential concern in this analysis is omitted variable bias. The
dataset only contains information about the resumes that were
submitted, but employers may respond to other factors that are not
observed in the data.

For example, the dataset does not include information about applicant
personality, interview performance, or professional networks. If
employers respond to these factors and they are correlated with the
variables included in the model, the estimated coefficients may partly
capture those omitted influences.

Because these variables are not observed in the dataset, they cannot
be directly controlled for in the regression model. This limitation
should be considered when interpreting the results.

## Conclusion

This notebook estimated a baseline econometric model to examine how
resume characteristics are associated with employer callback outcomes.

The Linear Probability Model provides an estimate of how factors such
as resume quality, work experience, and applicant background relate to
the probability of receiving a callback. Robust standard errors were
used to account for heteroskedasticity in the binary outcome variable.

An interaction term was also included to test whether the effect of
resume quality differs across racial groups. This type of analysis
helps explore whether certain resume characteristics have different
effects for different applicants.

These results provide a foundation for interpreting the economic
mechanisms that may influence employer responses to job applications.